# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
# Local, offline setup — uses only this workspace
import json, logging, sqlite3, subprocess, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

START = Path.cwd().resolve()
ROOT = next((path for path in (START, *START.parents)
             if (path / "data" / "raw" / "flasheats.db").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root above the working directory")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from challenges.notebook_support import (
    build_order_timeline, class7_order_model, lateness_view, load_context,
    normalize_ticket_category,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
ctx = load_context()
print("Workspace:", ROOT)
print("Published pipeline gate:", ctx["quality_report"]["gate"])
RAW = ROOT / "data" / "raw"

Workspace: /home/subaru/fde-work/FDE-Assignment-2
Published pipeline gate: WARN


In [2]:
orders=ctx["orders"]; customers=ctx["customers"]; restaurants=ctx["restaurants"]; drivers=ctx["drivers"]
tickets=ctx["support_tickets"]; customer_actions=ctx["customer_app_actions"]
interventions=ctx["order_interventions"]; outcomes=ctx["order_outcomes"]
order_events=ctx["order_events"]; restaurant_status=ctx["restaurant_status"]
model_brief=ctx["model_brief"]; canonical=lateness_view(ctx["journey"])
print("raw orders",orders.shape,"canonical",canonical.shape,"actions",customer_actions.shape,
      "interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

raw orders (1603, 13) canonical (1600, 74) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

---

### Completed answer

The executed timelines use **O00003** (delivered on time), **O00001** (delivered late), and **O00781** (intervention). Each timeline unions observed order-platform, driver-telemetry, app, support, restaurant-status and intervention timestamps, then sorts them. Outcome rows have no event timestamp and therefore are not fabricated as timeline events. Duplicate descriptions from different systems are retained because they are corroborating observations, not duplicates at the same source grain.


In [3]:
# Challenge 1 — three evidence-preserving timelines
base_orders = canonical
late_order = base_orders.loc[base_orders.late, "order_id"].iloc[0]
on_time_order = base_orders.loc[base_orders.ldr_eligible & ~base_orders.late, "order_id"].iloc[0]
intervention_order = interventions.order_id.iloc[0]
selected = {"on_time":on_time_order, "late":late_order, "intervention":intervention_order}
print(selected)
for journey_type, order_id in selected.items():
    print(f"\n{journey_type.upper()} — {order_id}")
    display(build_order_timeline(order_id, ctx))

{'on_time': 'O00003', 'late': 'O00001', 'intervention': 'O00781'}

ON_TIME — O00003


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer:C0855,order_events
1,2026-08-01 19:35:00.002443,assigned,driver:D039,driver_telemetry
2,2026-08-01 19:44:37.017130,gps_ping,driver:D039,driver_telemetry
3,2026-08-01 19:48:00.000000,preparing,restaurant,restaurant_status
4,2026-08-01 19:54:14.031817,gps_ping,driver:D039,driver_telemetry
5,2026-08-01 19:57:50.820366,picked_up,driver:D039,driver_telemetry
6,2026-08-01 19:57:50.820366,PICKED_UP,driver:D039,order_events
7,2026-08-01 20:03:51.046504,gps_ping,driver:D039,driver_telemetry
8,2026-08-01 20:13:28.061191,delivered,driver:D039,driver_telemetry
9,2026-08-01 20:13:28.061191,DELIVERED,driver:D039,order_events



LATE — O00001


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer:C0168,order_events
1,2026-08-13 13:01:09.659644,assigned,driver:D103,driver_telemetry
2,2026-08-13 13:08:00.000000,DRIVER_REASSIGNMENT,dispatch,interventions
3,2026-08-13 13:14:42.222353,gps_ping,driver:D103,driver_telemetry
4,2026-08-13 13:21:00.000000,ETA_VIEWED,customer,customer_app
5,2026-08-13 13:28:14.785062,gps_ping,driver:D103,driver_telemetry
6,2026-08-13 13:35:49.825561,picked_up,driver:D103,driver_telemetry
7,2026-08-13 13:35:49.825561,PICKED_UP,driver:D103,order_events
8,2026-08-13 13:41:47.347771,gps_ping,driver:D103,driver_telemetry
9,2026-08-13 13:55:19.910481,gps_ping,driver:D103,driver_telemetry



INTERVENTION — O00781


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer:C0103,order_events
1,2026-08-22 12:40:42.931820,assigned,driver:D103,driver_telemetry
2,2026-08-22 12:54:00.000000,ETA_VIEWED,customer,customer_app
3,2026-08-22 12:57:00.000000,PRIORITY_DISPATCH,operations,interventions
4,2026-08-22 13:04:00.000000,ready,restaurant,restaurant_status
5,2026-08-22 13:07:42.051142,gps_ping,driver:D103,driver_telemetry
6,2026-08-22 13:11:17.309762,picked_up,driver:D103,driver_telemetry
7,2026-08-22 13:11:17.309762,PICKED_UP,driver:D103,order_events
8,2026-08-22 13:22:41.170463,gps_ping,driver:D103,driver_telemetry
9,2026-08-22 14:01:40.289785,delivered,driver:D103,driver_telemetry


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

---

### Completed answer

| Table | Primary key | Important foreign keys | Grain |
|---|---|---|---|
| customers | customer_id | — | one customer |
| canonical orders | order_id | customer_id, restaurant_id, driver_id | one business order |
| customer_app_actions | action_id | order_id, customer_id | one app action |
| support_tickets | ticket_id (has one duplicate requiring quarantine) | order_id | one ticket record |
| interventions | intervention_id | order_id | one intervention |
| outcomes | order_id | order_id → orders | one supplied outcome per order |

The project model centers stable entities and aggregates one-to-many children before joining. This prevents fan-out/double counting and directly supports customer → order → interaction/intervention/outcome questions. Mirroring source tables would preserve storage design, not the business workflow, and would force every consumer to rediscover the grain.


In [4]:
# Challenge 2 — canonical model contract
model_contract = pd.DataFrame([
    ["customers","customer_id","—","one customer"],
    ["canonical orders","order_id","customer_id, restaurant_id, driver_id","one business order"],
    ["customer_app_actions","action_id","order_id, customer_id","one app action"],
    ["support_tickets","ticket_id*","order_id","one ticket record (*one duplicate ID quarantined)"],
    ["interventions","intervention_id","order_id","one intervention"],
    ["outcomes","order_id","order_id → orders","one supplied outcome per order"],
], columns=["Table","Primary key","Important foreign keys","Grain"])
display(model_contract)
print("Join policy: aggregate every one-to-many child to unique order_id before left joining.")

,Table,Primary key,Important foreign keys,Grain
0,customers,customer_id,—,one customer
1,canonical orders,order_id,"customer_id, restaurant_id, driver_id",one business order
2,customer_app_actions,action_id,"order_id, customer_id",one app action
3,support_tickets,ticket_id*,order_id,one ticket record (*one duplicate ID quarantined)
4,interventions,intervention_id,order_id,one intervention
5,outcomes,order_id,order_id → orders,one supplied outcome per order


Join policy: aggregate every one-to-many child to unique order_id before left joining.


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

---

### Completed answer

The order-level table has 1,600 unique orders. **163 of 843 late orders** had a linked support ticket. **430 orders** received an intervention. The most common intervention is **DRIVER_REASSIGNMENT (155)**, followed by restaurant contact 116, priority dispatch 95 and customer credit 64. “Frustrated” is explicitly defined as a linked support ticket or `CANCEL_ATTEMPTED`; **149** such journeys had no intervention, and **122** of them were late. Example IDs are shown by the executed cell. Missing activity is treated as zero only because each required source was present.


In [5]:
# Challenge 3 — interaction → intervention → timestamp-derived outcome
order_model = class7_order_model(ctx)
assert len(order_model) == order_model.order_id.nunique() == 1600
display(order_model[["order_id","customer_id","support_opened","cancel_attempted",
    "intervention_count","intervention_types","final_status","late_flag","delay_min"]].head())
frustrated = order_model[(order_model.support_opened | order_model.cancel_attempted) & order_model.intervention_count.eq(0)]
answers = {
    "late_orders_with_support": int((order_model.late_flag.eq(True) & order_model.support_opened).sum()),
    "orders_with_intervention": int(order_model.intervention_count.gt(0).sum()),
    "most_common_intervention": interventions.intervention_type.value_counts().idxmax(),
    "frustrated_without_intervention": len(frustrated),
    "frustrated_without_intervention_and_late": int(frustrated.late_flag.eq(True).sum()),
}
print(answers); display(interventions.intervention_type.value_counts().rename("count").to_frame())
display(frustrated[["order_id","support_opened","cancel_attempted","late_flag","delay_min"]].head(10))

,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,False,False,1,"(DRIVER_REASSIGNMENT,)",delivered,True,10.817265
1,O00002,C0043,False,False,0,(),delivered,True,3.969842
2,O00003,C0855,False,False,0,(),delivered,False,-4.414680
3,O00004,C0229,False,False,0,(),cancelled,<NA>,NaN
4,O00005,C0040,False,False,0,(),delivered,True,21.021512


{'late_orders_with_support': 163, 'orders_with_intervention': 430, 'most_common_intervention': 'DRIVER_REASSIGNMENT', 'frustrated_without_intervention': 149, 'frustrated_without_intervention_and_late': 122}


,count
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


,order_id,support_opened,cancel_attempted,late_flag,delay_min
36,O00037,True,False,True,15.452263
39,O00040,True,False,True,25.553780
40,O00041,True,False,True,5.395879
41,O00042,True,False,True,5.667206
48,O00049,True,False,True,1.752324
72,O00073,True,False,True,13.553937
82,O00083,True,False,True,2.369654
89,O00090,True,False,True,20.258256
115,O00116,True,False,True,12.793501
117,O00118,True,False,True,8.579891


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

---

### Completed answer

| Metric | Formula / grain | Measured value | KPI link |
|---|---|---:|---|
| Late Delivery Rate | late eligible delivered / eligible delivered; order | 843/1,495 = 56.39% | project outcome |
| Support-contact rate | orders with linked ticket / all orders; order | 198/1,600 = 12.38% | customer-friction signal |
| Cancel-attempt rate | orders with CANCEL_ATTEMPTED / all orders; order | 10/1,600 = 0.63% | severe customer-friction signal |
| Intervention reach | orders with ≥1 intervention / all orders; order | 430/1,600 = 26.88% | operational response |
| Median creation→pickup | median valid minutes; order | 26.14 min | workflow/driver metric |

The rates use order grain; raw event counts are not denominators. LDR remains provisional pending KPI-owner approval.


In [6]:
# Challenge 4 — compact project scorecard
metric_table = pd.DataFrame([
    ["Late Delivery Rate","late eligible / eligible delivered","order",843/1495,"Outcome; project KPI"],
    ["Support-contact rate","orders with linked ticket / all orders","order",198/1600,"Customer friction"],
    ["Cancel-attempt rate","orders with CANCEL_ATTEMPTED / all orders","order",10/1600,"Severe customer friction"],
    ["Intervention reach","orders with intervention / all orders","order",430/1600,"Operational response"],
    ["Median creation→pickup","median valid minutes","order",canonical.creation_to_pickup_min.median(),"Workflow duration (minutes)"],
], columns=["Metric","Formula","Grain","Value","KPI relationship"])
display(metric_table.round({"Value": 4}))

,Metric,Formula,Grain,Value,KPI relationship
0,Late Delivery Rate,late eligible / eligible delivered,order,0.5639,Outcome; project KPI
1,Support-contact rate,orders with linked ticket / all orders,order,0.1238,Customer friction
2,Cancel-attempt rate,orders with CANCEL_ATTEMPTED / all orders,order,0.0062,Severe customer friction
3,Intervention reach,orders with intervention / all orders,order,0.2688,Operational response
4,Median creation→pickup,median valid minutes,order,26.1400,Workflow duration (minutes)


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

---

### Completed answer

| Question | Evidence | Business meaning / non-claim |
|---|---|---|
| Support vs no support | 82.74% vs 52.39% late; mean delay 10.99 vs 1.93 min | tickets identify distressed journeys; they do not cause delay |
| Intervention vs none | 56.44% vs 56.37% late | no observed aggregate improvement; confounding by indication prevents effect claims |
| Lowest intervention-type late rate | priority dispatch 44/86 = 51.16% | useful follow-up segment; not proof priority dispatch works |
| Largest late-order contributor | R050: 22/35 late (62.86%) | prioritize review; volume and case mix matter |
| Support + intervention + still late | 44 orders | service recovery did not coincide with on-time outcome; intervention timing/selection matter |

All rates use timestamp-derived late labels and the eligible-delivery denominator; supplied outcome flags remain contextual.


In [7]:
# Challenge 5 — five order-grain investigations
eligible_model = order_model[order_model.ldr_eligible]
support_compare = eligible_model.groupby("support_opened").agg(orders=("order_id","count"), late=("late","sum"),
    late_rate=("late","mean"), mean_delay_min=("delay_min","mean"), median_delay_min=("delay_min","median"))
intervention_compare = (eligible_model.assign(intervention=eligible_model.intervention_count.gt(0))
    .groupby("intervention").agg(orders=("order_id","count"), late=("late","sum"), late_rate=("late","mean"), mean_delay_min=("delay_min","mean")))
intervention_type = (interventions.merge(order_model[["order_id","ldr_eligible","late","delay_min"]], on="order_id", validate="many_to_one")
    .query("ldr_eligible").groupby("intervention_type").agg(orders=("order_id","nunique"), late=("late","sum"), late_rate=("late","mean"), mean_delay_min=("delay_min","mean")).sort_values("late_rate"))
restaurants_late = (eligible_model.groupby("restaurant_id").agg(eligible_orders=("order_id","count"), late_orders=("late","sum"), late_rate=("late","mean"))
    .merge(restaurants[["restaurant_id","restaurant_name"]], on="restaurant_id").sort_values(["late_orders","late_rate"], ascending=False).head(10))
still_late = order_model[order_model.support_opened & order_model.intervention_count.gt(0) & order_model.late_flag.eq(True)]
for label, table in (("A Support",support_compare),("B Intervention",intervention_compare),("C Intervention type",intervention_type),("D Restaurants",restaurants_late)):
    print(label); display(table)
print("E support + intervention + still late:", len(still_late)); display(still_late[["order_id","delay_min","intervention_types"]].head(10))

A Support


,orders,late,late_rate,mean_delay_min,median_delay_min
support_opened,,,,,
False,1298,680,0.523883,1.933194,0.498471
True,197,163,0.827411,10.987702,11.433097


B Intervention


,orders,late,late_rate,mean_delay_min
intervention,,,,
False,1091,615,0.563703,3.084409
True,404,228,0.564356,3.239538


C Intervention type


,orders,late,late_rate,mean_delay_min
intervention_type,,,,
PRIORITY_DISPATCH,86,44,0.511628,2.694266
RESTAURANT_CONTACT,108,57,0.527778,3.215360
DRIVER_REASSIGNMENT,147,85,0.578231,2.755000
CUSTOMER_CREDIT,63,42,0.666667,5.155914


D Restaurants


,restaurant_id,eligible_orders,late_orders,late_rate,restaurant_name
49,R050,35,22,0.628571,Restaurant 050
23,R024,25,20,0.800000,Restaurant 024
50,R051,29,20,0.689655,Restaurant 051
3,R004,30,20,0.666667,Restaurant 004
22,R023,32,20,0.625000,Restaurant 023
29,R030,38,20,0.526316,Restaurant 030
44,R045,29,19,0.655172,Restaurant 045
10,R011,31,19,0.612903,Restaurant 011
17,R018,27,18,0.666667,Restaurant 018
24,R025,28,18,0.642857,Restaurant 025


E support + intervention + still late: 44


,order_id,delay_min,intervention_types
57,O00058,2.695561,"(CUSTOMER_CREDIT,)"
78,O00079,1.535014,"(PRIORITY_DISPATCH,)"
93,O00094,19.561637,"(DRIVER_REASSIGNMENT,)"
161,O00162,8.415848,"(CUSTOMER_CREDIT,)"
173,O00174,15.106535,"(PRIORITY_DISPATCH,)"
197,O00198,35.234569,"(RESTAURANT_CONTACT,)"
217,O00218,10.288430,"(RESTAURANT_CONTACT,)"
279,O00280,5.096155,"(DRIVER_REASSIGNMENT,)"
287,O00288,19.086669,"(RESTAURANT_CONTACT,)"
300,O00301,15.785165,"(CUSTOMER_CREDIT,)"


# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

---

### Completed answer

**PROJECT KPI: Reduce provisional Late Delivery Rate**

→ **Outcome metrics:** LDR 56.39%; >10-minute rate 23.34%  
→ **Workflow/customer metrics:** creation→pickup median 26.14 min; support-contact 12.38%; cancel-attempt 0.63%  
→ **Intervention metrics:** intervention reach 26.88%; type/timing mix; directly controllable are assignment/reassignment, priority dispatch, restaurant contact and credit policy—not traffic/weather or the outcome itself  
→ **Sources/events:** canonical orders + order events + driver telemetry + app actions + tickets + interventions + Dispatch + contextual supplied outcomes

Core relationships are customer 1→many orders and order 1→many actions/tickets/interventions/events, with one canonical order outcome. The largest modelling limitation is the missing **driver-arrived-at-restaurant** event: creation→pickup cannot separate driver approach from restaurant wait. Instrument a timestamped, timezone-aware, geofence/accuracy-governed arrival event next, plus explicit intervention eligibility/decision timestamps for later causal evaluation.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.